In [0]:
EH_NAMESPACE = '<namespace>.servicebus.windows.net'
EH_NAME = '<event-hub-name>'
SERVICE_CREDENTIAL = '<service-credential>'

CATALOG = spark.sql('SELECT current_catalog()').first()[0]
SCHEMA = 'iss_tracker'

BRONZE = f'{SCHEMA}.bronze_events'
SILVER = f'{SCHEMA}.silver_positions'
GOLD = f'{SCHEMA}.gold_positions'

CKPT_ROOT = f'/Volumes/{CATALOG}/{SCHEMA}/pipeline_state/checkpoints'
CKPT_BRONZE = f'{CKPT_ROOT}/bronze_v1'
CKPT_SILVER = f'{CKPT_ROOT}/silver_v2'

KAFKA_OPTIONS = {
    'kafka.bootstrap.servers': f'{EH_NAMESPACE}:9093',
    'subscribe': EH_NAME,
    'databricks.serviceCredential': SERVICE_CREDENTIAL
}

print('CATALOG    :', CATALOG)
print('Hub        :', EH_NAMESPACE, '/', EH_NAME)
print('Checkpoints:', CKPT_ROOT)

In [0]:
from pyspark.sql.functions import col, current_timestamp

def run_bronze_stream():
    raw = (spark.readStream.format('kafka')
           .options(**KAFKA_OPTIONS)
           .option('startingOffsets', 'earliest')
           .load())
    bronze = raw.select(
        col('value').cast('string').alias('raw_json'),
        col('partition'),
        col('offset'),
        col('timestamp').alias('broker_time'),
        current_timestamp().alias('processed_at')
    )

    query = (bronze.writeStream
             .format('delta')
             .option('checkpointLocation', CKPT_BRONZE)
             .trigger(availableNow=True)
             .toTable(BRONZE))
    query.awaitTermination()
    return query

In [0]:
q = run_bronze_stream()
print('Query active: ', q.isActive)

bronze = spark.table(BRONZE)
print('Bronze rows: ', bronze.count())
bronze.orderBy(col('offset').desc()).show(5, truncate=90)

In [0]:
before = spark.table(BRONZE).count()
run_bronze_stream()
after = spark.table(BRONZE).count()
print(f'before = {before}, after = {after}, added = {after - before}')

b = spark.table(BRONZE)
dupes = b.count() - b.dropDuplicates(['partition', 'offset']).count()
print(f'Duplicate (partition, offset) pairs: {dupes}')